# <center>智能体项目开发必备基础:FastAPI Foundations</center>

&emsp;&emsp;今天我们聚焦的是一件很多同学都在体感上卡住的事情——怎么把一份"能在自己电脑上跑通的 Python 脚本",升级成一个"别人能用、能被前端调用、能被其他系统对接"的后端服务。这中间隔着的不是一段代码,而是一整个工程领域:**后端服务 + API**。我们今天用最短的路径把这道鸿沟跨过去,工具选 2026 年 AI 时代 Python 后端开发的默认答案—— `FastAPI`。

<div align=center><font size=2 color=#999999>从本地脚本到对外服务:FastAPI 架起的那座桥</font></div>
<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/fastapi-foundations/2026-05-15/img-58e8400b.png" width=80%></div>

<br>

&emsp;&emsp;上面这张图就是本课要跨越的距离——左边是能在本地跑通的 Python 脚本,右边是任何人、任何语言都能调用的后端服务。

&emsp;&emsp;结合一线工程实践与当下的 AI 工程生态,我们会重点抓住四件事:

&emsp;&emsp;第一,看清楚 `FastAPI` 到底站在整个应用架构里的哪个位置,理解一个后端服务不是孤岛,而是调用链上的一环。

&emsp;&emsp;第二,把"接口规范"这件事讲透,从"没有规范的联调痛苦"到 `OpenAPI` 标准,再到 `Pydantic` 把规范自动化的底层机制。

&emsp;&emsp;第三,动手跑通两个完整案例——普通的"文本分析后端"和 AI 时代典型的"流式 LLM 接口",把 `async`、`CORS`、`StreamingResponse` 这些绕不开的概念一次性打通。

&emsp;&emsp;第四,留下一份可以长期翻阅的速查表,把后端开发常遇到的 `422`、`404`、`CORS` 红屏、`async` 决策等场景都沉淀下来。

&emsp;&emsp;为了把这些内容讲清楚,接下来我们会按一条"从宏观到落地"的主线展开:先看应用架构的全景,建立心里有数;再讲多人协作如何催生接口规范、规范又如何被 `Pydantic` 自动化;然后是 `FastAPI` 的立体认知——向内看组成、横切看能力、向外看接口形态;接下来是环境准备和两个完整案例;最后是小结与速查表。

<div align=center><font size=2 color=#999999>课程总体概览:从应用架构、接口规范到两个 FastAPI 实战案例</font></div>
<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/fastapi-foundations/2026-05-15/img-db02db61.png" width=80%></div>

<br>

&emsp;&emsp;先把这张路线图放在这里,后面每进入一章,都能知道自己正在从"脚本"走向"可对外调用的后端 API"的哪一步。我们接下来先从"一个真实后端服务在整个系统里的位置"讲起。


## 1. 应用架构全景

&emsp;&emsp;本章的目的是在写第一行 `FastAPI` 代码之前,先把"我写的这个服务到底处于整个应用的哪个位置"这件事看清楚。这不是抽象概念的铺垫,而是直接决定了后面所有代码的写法——如果我们心里没有这张全景图,写出来的服务很容易变成"能跑但放不进真实系统"的孤岛。本章会按三个尺度递进:先看宏观三层骨架,再放大到一次真实请求经过的 13 个节点,最后把"脚本"和"服务"这两种形态的本质差别列清楚。


### 1.1 后端基础设施的三层骨架

&emsp;&emsp;我们打开任何一个熟悉的 App,比如微信、抖音、美团,把它们的服务端架构从抽象层面压缩一下,基本上都会归到同样的三层结构上:接入、逻辑、存储。这不是某个公司的特殊设计,而是互联网服务端工程十多年沉淀出来的通用骨架。

<div align=center><font size=2 color=#999999>应用三层架构示意图:接入层负责让请求进得来,逻辑层负责处理业务,存储层负责持久化</font></div>
<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/fastapi-foundations/2026-05-15/img-a627ac51.png" width=80%></div>

<br>

&emsp;&emsp;这三层各自承担的职责是这样的。**接入层**负责"让请求进得来",包含 `TLS/HTTPS` 终止(处理证书和加解密,让后端应用拿到已解密的请求)、限流、鉴权等能力,通常由 `Nginx`、`API Gateway` 这类入口组件承担;**逻辑层**拆成几十上百个独立的小服务,每个服务专注做一件事(订单、用户、支付、消息等),这是我们这节课真正要站的位置;**存储层**负责把业务数据放到合适的介质里,`MySQL`、`Redis`、对象存储各司其职。



### 1.2 外卖下单背后的 13 个节点

&emsp;&emsp;三层骨架是宏观视角。我们再放大一档:一次真实的用户请求,到底会穿过哪些节点?选一个日常场景——打开外卖 App,点"下单",几秒钟后骑手出现在地图上,这中间发生了什么?

<div align=center><font size=2 color=#999999>外卖下单背后的 13 节点调用链:客户端、基础设施、后端业务服务三类职责并存</font></div>
<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/fastapi-foundations/2026-05-15/img-d82a900d.png" width=80%></div>

<br>

&emsp;&emsp;这 13 个节点按职责可以分成三类。第一类是**客户端**(节点 1、11、13),手机 App 或浏览器里的代码,和 `FastAPI` 无关;第二类是**基础设施**(节点 2、3、8),`DNS`、`API 网关`、`事件总线`等,由专门的中间件承担(`Nginx`、`Kafka` 等),也不需要业务框架;第三类才是**后端业务服务**(节点 4、5、6、7、9、10、12),`BFF`、库存、订单、支付、调度、消息、位置——<font color=red>这 7 个节点都可以用 `FastAPI` 来写</font>。

&emsp;&emsp;也就是说,13 个节点里有过半数都是 `FastAPI` 的用武之地。它们本质上做的是同一件事:**接 JSON、做处理、回 JSON**。区别只在中间"做处理"这一步——订单服务做的是写订单库,库存服务做的是扣库存,AI 服务做的是调用 `LLM`。今天我们会用一个"文本分析服务"作为最小可跑示例,"做处理"那一步换成字数统计和词频 Top 5,它的整体骨架和上述任意一种业务服务完全相同。

&emsp;&emsp;<font color=red>注意:</font>入门课要的不是"理解全部 13 个节点",而是建立心里有数——写出来的服务不是孤岛,是这条链上的一环;学会写其中一种,其他后端业务服务的写法本质上一致。


### 1.3 脚本到服务的本质区别

&emsp;&emsp;那么,要把一个普通 Python 函数(比如 `analyze_text(text: str) -> dict`,对一段中文做字数统计和词频 Top 5)升级成上面那种链路里的一环,到底要补什么?直觉里能想到的几种方案——把 `demo.py` 直接发给朋友、远程帮跑一次、传到微信里发一份——都不太对,问题在于它们都还是"一次性运行",而不是"能被任意调用方持续调用"。我们把这两种形态的核心差异整理成一张对照表。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>脚本与服务的本质差异</font></p>
<div class="center">

| 维度 | 脚本(`python demo.py`) | 服务(本课要学的) |
|---|---|---|
| **谁触发执行** | 自己在终端敲命令 | 任何人发 `HTTP` 请求都能触发 |
| **生命周期** | 跑完就退出 | 7×24 小时常驻 |
| **多端复用** | 需要 Python 环境 | 浏览器 / `curl` / 任何语言都能调 |
| **协议规范** | 函数签名只写代码人自己看 | `HTTP+JSON`,跨平台标准 |
| **输入校验** | 自己写 `if/else` | 框架统一处理 |
| **错误返回** | `raise Exception` | `HTTP 状态码` + 错误体 |

</div>

&emsp;&emsp;从脚本到服务,<font color=red>最大的变化不是代码量,而是"被任意调用方使用"这个新的存在方式</font>。代码不再只写给作者看,而是写给外面的世界对接的接口。既然要被外部对接,就免不了和别人协作。下一章我们先把这件事看清楚——一个真实的后端是怎么被多人协作出来的,以及"协作"会催生出一个绕不过的概念:**接口规范**。


## 2. 多人协作与接口规范

&emsp;&emsp;上一章我们把"脚本到服务"的本质差别讲清楚了——核心是服务要被外部调用方消费。但一旦进入"被消费"的阶段,就不再是一个人写完就结束的事情,后端输出的接口会立刻被前端、测试、其他后端服务使用。本章我们沿着这条"协作"的线走下去,看清楚为什么**接口规范**是绕不开的,以及 `OpenAPI` 和 `Pydantic` 是怎么把这件事工业化的。


### 2.1 软件不是一个人写出来的

&emsp;&emsp;稍有规模的 App,背后通常都有这样一支队伍:产品提需求、设计出 UI、前端写页面、后端写接口、测试做验证、运维管部署。这就是软件开发生命周期(`SDLC`)的标准模型。

<div align=center><font size=2 color=#999999>软件开发协作流程:本课所学的 FastAPI 处于"开发-后端"这个位置</font></div>
<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/fastapi-foundations/2026-05-15/img-a9428e0c.png" width=80%></div>

<br>

&emsp;&emsp;我们今天所学的 `FastAPI`,处于"**开发-后端**"这个位置。但要意识到这不是孤立工作——后端的输出,也就是接口,会立刻被前端、`QA` 和其他后端服务使用。入门课不需要覆盖全套 `SDLC`,但有一件事必须先看清楚:**前后端联调到底在哪个环节最容易出问题**。


### 2.2 没有接口规范的联调长什么样

<div align=center><font size=2 color=#999999>前后端协作场景:一个字段名、一个错误码,都可能在联调里反复消耗时间</font></div>
<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/fastapi-foundations/2026-05-15/img-c633632b.png" width=80%></div>

<br>

&emsp;&emsp;我们把典型的"口头约定"场景还原一下。前端工程师发来消息:"登录接口怎么调?传什么字段?"后端回:"`POST`,传 `username` 和 `password`,回 `token`。"半小时后前端说跑不通——后端一看,对方传的是 `user_name`,多了个下划线。口头交代的字段名,一个字符就能出错。

&emsp;&emsp;再过一会儿:"传了错误的密码,接口返回 200 OK,我前端没法判断登录成没成。"——后端心想"是 200,`body` 里有 `error` 字段",但这个从来没说过。第二天另一个前端同事也来问同样的问题,只能从头再说一遍。

&emsp;&emsp;<font color=red>靠口头约定维护的接口,每变更一次、每来一个新对接方,都要重新同步一次</font>——一个十人团队,一周能因此浪费十几个小时。这不是某个团队水平差,而是没有把"约定"沉淀到一份可共享的载体上必然的结果。


### 2.3 接口规范:让协作不靠记性

<div align=center><font size=2 color=#999999>接口规范示意:把"接受什么、返回什么、可能出什么错"写到一份大家都能看的文档里</font></div>
<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/fastapi-foundations/2026-05-15/img-69609a48.png" width=80%></div>

<br>

&emsp;&emsp;破解上一节那个困局的办法朴素得令人意外——**把约定写下来**。把"这个接口接受什么、返回什么、可能出什么错"写在一份大家都能看的文档里,这份文档就是**接口规范**(在工程界也称为 "`API 契约` / `API Contract`",强调"双方约定遵守"的语义)。

&emsp;&emsp;有了规范之后,前端按规范写代码,后端按规范实现,`QA` 按规范测试,对接方按规范接入。重复的口头问题大幅减少,协作摩擦降下来了。但接口规范本身也得有标准——不能随便画个表格就行,得用一种**机器和人都看得懂的格式**,这样工具才能自动校验、自动生成代码、自动渲染文档。这就引出了下一个层次:通用的规范描述格式。


### 2.4 OpenAPI:通用的接口规范格式

&emsp;&emsp;`OpenAPI`(前身叫 `Swagger`)就是这个工业标准格式。它用 `JSON` 或 `YAML` 描述接口的请求、响应、错误码、字段类型等等。我们看一个最小的片段来建立直观印象。下面这段 `YAML` 描述了一个 `/login` 接口:它接受 `POST`,请求体里有两个必填字段 `username` 和 `password`(后者长度至少 6),成功返回 200 带 `token`,密码错误返回 401。


```yaml
    paths:
      /login:
        post:
          requestBody:
            content:
              application/json:
                schema:
                  type: object
                  required: [username, password]
                  properties:
                    username: { type: string }
                    password: { type: string, minLength: 6 }
          responses:
            '200':
              content:
                application/json:
                  schema:
                    type: object
                    properties:
                      token: { type: string }
            '401':
              description: 密码错误
```


&emsp;&emsp;这种 `YAML` 一旦写好,就能喂给前端工具自动生成 `TypeScript` 类型定义、喂给后端工具自动生成 `Python/Java` 客户端 `SDK`、喂给 `Swagger UI` 自动渲染成可点击调用的网页文档、喂给 `Postman` 自动导入成接口集合。`OpenAPI` 是目前最通用的接口规范格式,主流后端框架、云厂商、`API` 平台基本都支持它。

&emsp;&emsp;<font color=red>但有一个问题——手写 `OpenAPI YAML` 太痛苦了</font>。一个有 20 个接口的服务,`YAML` 能写到几千行。改代码 5 分钟,改 `YAML` 半小时,最终大家都懒得维护,文档和代码不同步成了死结。**这就是 `FastAPI` 出场的地方**。


### 2.5 Pydantic:用 Python 类型提示当规范

&emsp;&emsp;`FastAPI` 的核心创意是:**让人写 Python 代码,自动生成 `OpenAPI` 规范**。具体做法是借助 `Pydantic` 这个库——它在 Python 数据验证生态中是通用的数据建模库,核心动作是"**用 Python 类型提示来声明数据结构,自动获得校验 + 序列化 + 文档**"。

<div align=center><font size=2 color=#999999>Pydantic 核心机制:从类型提示到校验、序列化与 OpenAPI 规范的一体化</font></div>
<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/fastapi-foundations/2026-05-15/img-6545edc1.png" width=80%></div>

<br>

&emsp;&emsp;这是 `Pydantic` 在本课的**第一次出场**。我们用一个登录请求的最小示例看它的核心语法——一个继承自 `BaseModel` 的 class,字段写好 Python 类型提示,约束条件用 `Field` 附加。


In [ ]:
from pydantic import BaseModel, Field

class LoginRequest(BaseModel):
    username: str
    password: str = Field(min_length=6)


&emsp;&emsp;就这么一个 class——没有任何 `if/else` 校验代码,但 `Pydantic` 已经做完了下面这些事:`username` 必须是字符串,否则报错;`password` 必须是字符串,且长度 ≥ 6,否则报错;错误被自动整理成清晰的 `JSON` 返回格式;这个数据结构会被自动转成 `OpenAPI Schema`。

&emsp;&emsp;<font color=red>重点</font>:`Pydantic` 并不只服务于 `FastAPI`。`LangChain`、`LlamaIndex`、`OpenAI SDK`、`LiteLLM` 等 Python AI 工程生态里,大量接口、配置和结构化输出都会用到 `Pydantic` 或 `JSON Schema` 这类数据约束机制。学好 `Pydantic` 是入场 Python 后端与 AI 工程的通用技能,而**不是 `FastAPI` 的附属品**。

&emsp;&emsp;到这里,整条逻辑链已经清楚:**协作的核心是接口规范,规范的标准是 `OpenAPI`,`OpenAPI` 的 Python 写法是 `Pydantic`**。下一章我们就来看清楚 `FastAPI` 这个把它们串起来的工具,到底由什么组成、能做什么、以及为什么在 AI 时代会成为高频选择。


## 3. FastAPI 立体认知

&emsp;&emsp;前两章我们一直没有正面介绍 `FastAPI` 本身。这一章我们从三个视角立体地把它认识清楚:向内看它由什么组成、横切看它提供哪些核心能力、向外看它写出来的接口最终呈现给消费方的样子。最后再解释为什么在 2026 年的 AI 时代,它已经从"一个可选框架"变成了非常高频的服务层选择。


### 3.1 向内:FastAPI 由什么组成

&emsp;&emsp;`FastAPI` 并不是一个独立从零实现的框架。它更准确的定位是:站在几个成熟 Python 库之上,把路由、中间件、数据校验和接口文档整合成一套统一、顺手的开发体验。

<div align=center><font size=2 color=#999999>FastAPI 内部组成:Starlette 提供路由、Pydantic 提供校验、OpenAPI 生成器负责文档</font></div>
<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/fastapi-foundations/2026-05-15/img-c887ffc7.png" width=80%></div>

<br>

&emsp;&emsp;三个核心组成部分各司其职。**`Starlette`** 提供路由和中间件,把"`HTTP` 请求"翻译成"调用哪个 Python 函数";**`Pydantic`** 提供数据校验,把"请求里的 `JSON`"翻译成"Python 对象",反过来也行;**`OpenAPI` 生成器**把代码自动转成 `OpenAPI YAML`。`FastAPI` 的价值不在于把这些能力全部重写一遍,而在于把成熟能力组合好,再提供一个统一、清晰、适合工程落地的开发者接口。


### 3.2 横切:五件套核心能力

&emsp;&emsp;组装好之后,开发者面对的是一个统一界面。打开任意一个 `FastAPI` 项目,反复出现的就是下面这五件套。

<div align=center><font size=2 color=#999999>FastAPI 五件套能力图:声明式路由、自动校验、自动文档、异步支持、中间件</font></div>
<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/fastapi-foundations/2026-05-15/img-c86683fb.png" width=80%></div>

<br>

&emsp;&emsp;第一件是**声明式路由**,一行 `@app.get("/users/{id}")` 就把 `HTTP` 路径绑定到 Python 函数;第二件是**自动校验**,参数声明 `Pydantic` 模型后,请求自动校验,错了自动 422;第三件是**自动文档**,代码即文档,浏览器打开 `/docs` 直接交互调试;第四件是**异步支持**,`async def` 路由原生支持,跑 `LLM` 这类慢请求不阻塞;第五件是**中间件与依赖机制**,`CORS`、认证、日志、限流等横切能力可以通过中间件、依赖注入或第三方库接入。入门课会把最常见的 `CORSMiddleware` 放到案例一末尾结合实际跨域场景来讲。


### 3.3 向外:接口最终长什么样

&emsp;&emsp;我们再换一个方向看——`FastAPI` 写出来的接口,最终呈现给消费方(前端、别的服务、`Postman`)的样子是什么?通常是 **`HTTP + JSON`**。下面这段是客户端(`curl` 或浏览器)和服务端一次完整往返的报文形态。


```http
    请求:
    POST /analyze HTTP/1.1
    Content-Type: application/json

    {"text": "FastAPI lets you write APIs faster than ever."}

    响应:
    HTTP/1.1 200 OK
    Content-Type: application/json

    {"word_count": 9, "char_count": 48}
```


&emsp;&emsp;这里有一个很重要的观察:不管后端是 Python、`Node.js` 还是 `Go`,`HTTP + JSON` 是跨语言的统一接口。**对前端来说,通常不需要关心后端是哪种语言写的;它真正依赖的是接口路径、请求字段、响应结构和错误约定**。这正是 `OpenAPI` 能成为标准的原因——它描述的是"约定",而不是"语言"。


### 3.4 为什么 AI 时代 FastAPI 是高频选择

&emsp;&emsp;如果说 5 年前后端接口选 `FastAPI` 还要权衡,2026 年的 AI 工程里它已经是非常高频的选择。我把原因压成三条、并给出每条背后的量化直觉。

&emsp;&emsp;**第一,`LLM` 调用天然适合异步等待**。一次大模型调用通常以秒计,推理模型还可能到几十秒甚至分钟级。传统同步框架如果采用 `gunicorn sync worker` 这类同步 worker 模式,一个 worker 同一时刻只能处理一个请求;假设单次请求耗时 15 秒,1 分钟大约只能处理 4 个。`FastAPI` 用 `async def` + `ASGI`,在等待上游返回时可以把控制权交还给事件循环,让同一个 worker 同时挂起更多等待型请求。这个差距在我们后面的"100 并发对照实验"里会有非常直观的体感。

&emsp;&emsp;**第二,结构化输出离不开 `Pydantic`**。让 `LLM` 返回严格的 `JSON` 结构(`OpenAI Structured Outputs` / `Anthropic Tool Use`)的工业标准是 **`JSON Schema`**。Python 里从类型提示生成 `JSON Schema` 最方便的工具就是 `Pydantic`——所以 AI 工程里 `Pydantic` 几乎成了通用标准。`FastAPI` 接口和 `LLM` 工具调用可以用**同一套 `Pydantic` 模型**,零额外学习成本。

&emsp;&emsp;**第三,Python AI 生态里 `FastAPI` 已经是非常常见的服务层选择**。例如 `LangServe`、`LlamaIndex` 相关 API 示例、`LiteLLM Gateway`、`vLLM` 的 `OpenAI` 兼容服务等,都能看到 `FastAPI`/`ASGI` 这套路线的身影。换句话说,学会 `FastAPI`,也是在看懂和复用很多 Python AI 开源项目服务层设计的入口。

&emsp;&emsp;到这里,`FastAPI` 的位置、能力和价值已经铺好了。接下来我们把第一个服务真正跑起来,用 5 分钟在浏览器里完成一次调用——从这一刻起,前面所有的概念都会被实证到屏幕上。


## 4. 环境准备:第一个能跑的 FastAPI 服务

&emsp;&emsp;上一章把 `FastAPI` 的位置、组成、能力讲清楚之后,本章我们把它真正跑起来。我会把整个过程拆成 8 个最小步骤,全部在本地机器完成,总时间 5-10 分钟。完成之后你手上会有:一个独立的 Python 虚拟环境、一份 6 行的 `analyzer.py`、一个在本地 8000 端口跑着的 `FastAPI` 服务,以及一份**浏览器里直接点就能调用的接口文档**。

&emsp;&emsp;<font color=red>注意:</font>本课全程 `macOS / Linux / Windows` 三平台都能跑。两套系统差异较大的命令(`venv` 激活、带 `~` 的 `cd`、`curl`、`bash` 脚本)会**并列给出两个代码块**;三平台写法一致的命令(`pip install`、`uvicorn`、`python -m http.server`)只写一份,不再重复。**Windows 学员请准备 `PowerShell 7`**(不是系统自带的 "Windows PowerShell 5.1"),可以通过 `winget install --id Microsoft.PowerShell` 安装,之后从开始菜单启动 "PowerShell"。文中出现的 `~/foo` 在 Windows 上等价于 `$HOME\foo`。


### 4.1 创建项目目录

&emsp;&emsp;<font color=red>第一步,</font>给我们的第一个 `FastAPI` 项目创建一个独立的目录。这一个目录会在整个案例一的后续步骤里持续使用——文本分析后端的所有代码都基于这一份 `analyzer.py` 层层叠加。**案例二会开一个独立的新项目**,到时另开一个目录即可。

&emsp;&emsp;`macOS / Linux`(终端)的命令是这样:


```bash
    mkdir text-analyzer-api
    cd text-analyzer-api
```


&emsp;&emsp;`Windows`(`PowerShell`)的命令等价:


```powershell
    mkdir text-analyzer-api
    cd text-analyzer-api
```


&emsp;&emsp;执行之后,终端会停留在刚创建的 `text-analyzer-api/` 目录下。所有后续的命令都默认在这个目录里执行。


### 4.2 确认 Python 版本

&emsp;&emsp;<font color=red>第二步,</font>确认本机 Python 版本不低于 3.10。本课后面会用到 `str | None` 这种联合类型语法,它是 Python 3.10 起才支持的。

&emsp;&emsp;`macOS / Linux`(终端):


```bash
    python3 --version
```


&emsp;&emsp;`Windows`(`PowerShell`):


```powershell
    py --version
```


&emsp;&emsp;看到 `Python 3.10.x` 或更高就可以继续。版本太老就去 `python.org` 装个新的(`Windows` 安装时记得勾选 "`Add Python to PATH`")。


### 4.3 创建并激活 venv

&emsp;&emsp;<font color=red>第三步,</font>给这个项目建一个独立的 `venv`(虚拟环境)。之所以每个项目都要单独一个 `venv`,是因为直接 `pip install` 到系统 Python,几个项目依赖冲突之后整个 Python 环境就会坏掉。<font color=red>每个项目一个 `venv`,装坏了删掉重来,对主环境零伤害</font>——这是 Python 开发的基本卫生习惯。

&emsp;&emsp;`macOS / Linux`(终端):


```bash
    python3 -m venv venv
    source venv/bin/activate
```


&emsp;&emsp;`Windows`(`PowerShell`):


```powershell
    py -m venv venv
    .\venv\Scripts\Activate.ps1
```


&emsp;&emsp;激活成功的标志是命令行前面多了 `(venv)` 前缀。

&emsp;&emsp;<font color=red>Windows 小坑:</font>第一次执行 `Activate.ps1` 如果报"在此系统上禁止运行脚本",先跑一次 `Set-ExecutionPolicy -Scope CurrentUser RemoteSigned`(只改当前用户,安全),再重试激活。如果偏好 `CMD` 不用 `PowerShell`,激活命令换成 `venv\Scripts\activate.bat`。


### 4.4 安装 fastapi[standard]

&emsp;&emsp;<font color=red>第四步,</font>安装 `FastAPI` 全家桶。保持 `venv` 激活状态,**三平台通用**命令如下:


```bash
    pip install "fastapi[standard]"
```


&emsp;&emsp;`[standard]` 是一个**全家桶标记**,会一并装上:`fastapi` 主体、`Pydantic`、`Starlette`、`uvicorn` 服务器、`httpx` 客户端、自动文档需要的 `jinja2`——**一行命令搞定所有依赖**。

&emsp;&emsp;> &emsp;**国内镜像源**:默认 `PyPI` 在国内可能慢。可以加 `-i https://pypi.tuna.tsinghua.edu.cn/simple` 用清华源,或全局配置 `pip config set global.index-url https://pypi.tuna.tsinghua.edu.cn/simple`。


### 4.5 写第一个 analyzer.py

&emsp;&emsp;<font color=red>第五步,</font>在 `~/text-analyzer-api/` 下创建 `analyzer.py`。<font color=red>注意:</font>这里的代码是要**保存到文件**的,不是在 Notebook 里直接运行的(因为后面需要用 `uvicorn` 加载这个文件)。我们用编辑器粘贴下面 6 行进去保存即可。


In [ ]:
from fastapi import FastAPI

app = FastAPI()


@app.get("/")
def read_root():
    return {"message": "Hello FastAPI"}


&emsp;&emsp;**就 6 行代码**。一个 `FastAPI` 应用最小能写到这个长度——它做了三件事:导入 `FastAPI` 类、实例化 `app`、用 `@app.get("/")` 装饰器把 `read_root` 函数绑定到根路径的 `GET` 请求。这已经是一个完整、能独立运行的服务。


### 4.6 启动 uvicorn

&emsp;&emsp;<font color=red>第六步,</font>启动服务器。保持在 `text-analyzer-api/` 目录、`venv` 激活状态,**三平台通用**:


```bash
    uvicorn analyzer:app --reload
```


&emsp;&emsp;这里的 `analyzer:app` 格式是 `文件名:对象名`(不带 `.py`),含义是"加载 `analyzer.py` 里那个叫 `app` 的对象"。`--reload` 开启开发模式,改代码自动重启服务。看到 `Application startup complete` 就是启动成功。

&emsp;&emsp;<font color=red>`uvicorn` 到底是什么?</font>它是一个 `ASGI` 服务器——工作是把网络层进来的 `HTTP` 请求转交给 `FastAPI app` 处理,再把响应写回去。`FastAPI` 只负责"怎么处理这个请求",`uvicorn` 负责"怎么和外面的网络对接"。


### 4.7 浏览器访问根路由

&emsp;&emsp;<font color=red>第七步,</font>打开浏览器,地址栏输入 `http://127.0.0.1:8000`。页面会显示一行干净的 `JSON`:


```json
    {"message": "Hello FastAPI"}
```


&emsp;&emsp;这一刻,浏览器的动作链是:发了 `GET` 请求 → `uvicorn` 收到 → `FastAPI` 找到 `read_root` 函数调用 → 把返回值序列化成 `JSON` 写回浏览器。到这里,一个完整的"请求-响应"闭环已经成立。


### 4.8 打开 /docs 看自动文档

&emsp;&emsp;<font color=red>第八步,</font>地址栏切到 `http://127.0.0.1:8000/docs`——浏览器会渲染出一个**可点击的接口文档界面**。这就是 **`Swagger UI`**,由 `FastAPI` 帮我们自动生成。回顾前面建立的那条逻辑链:

> &emsp;协作需要规范 → `OpenAPI` 是规范的标准格式 → `FastAPI` 把 `OpenAPI` 规范的生成自动化 → 自动渲染出 `Swagger UI`

&emsp;&emsp;我们写了 6 行 Python 代码,没写任何文档、没装任何文档工具、没配置任何东西,就得到了一份完整的可交互文档——<font color=red>这就是"规范自动化"的真实形态,这是我们本课贯穿线的第一次呼应</font>。

&emsp;&emsp;<font color=red>小提示:</font>在 `Swagger UI` 里点开 `GET /` 那一行的 "`Try it out`" → "`Execute`",下方会显示 200 OK + 响应体 + 完整的 `curl` 命令。对接方可以在这里自行调试大多数常见情况,不用每次都来问后端。

&emsp;&emsp;6 行 Hello World 还只是雏形。下一章我们把它扩展成一个**完整的应用**——前端、后端、接口完整跑通的文本分析器。


## 5. 案例一:文本分析后端

&emsp;&emsp;上一章我们用 6 行代码跑通了第一个 `FastAPI` 服务。本章我们把它扩展成一个**完整的应用**——后端暴露三个真实的业务接口,前端用一个 `HTML` 页面在浏览器里调用它们,还会遇到并解决"跨域"这个前后端联调里最常见的一堂必修课。走完这一章,你手里就有一个可以直接改造成任何业务的通用 API 模板。

<div align=center><font size=2 color=#999999>案例一应用全景:前端 HTML 页面调用 FastAPI 三路由(Body/Path/Query),返回 JSON</font></div>
<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/fastapi-foundations/2026-05-15/img-8438a73e.png" width=80%></div>

<br>


### 5.1 这一节要做什么

&emsp;&emsp;我们要做的是一个最小可用的"文本分析"后端,对外暴露 3 个接口,它们覆盖了 `FastAPI` 最重要的三种参数入口。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>案例一的三个接口</font></p>
<div class="center">

| 方法 | 路径 | 作用 | 参数入口 |
|---|---|---|---|
| `POST` | `/analyze` | 提交一段中文,返回字数 / 字符数 / 词频 Top 5 | Body |
| `GET` | `/history/{id}` | 按 ID 取历史记录 | Path |
| `GET` | `/search?keyword=xxx` | 按关键词搜索历史 | Query |

</div>

&emsp;&emsp;再加一个 `demo` 前端页面,能在浏览器里贴文本、点按钮、看结果——**这就是一个完整的应用**,前端 + 后端 + 接口联通。我们开始逐步写代码。


### 5.2 装中文分词工具 jieba

&emsp;&emsp;<font color=red>第一步,</font>装一个中文分词库。中文不像英文有空格分词,需要专门的库来切词。`jieba` 是 Python 生态中最成熟的中文分词工具——零配置、加载快、单条命令安装。

&emsp;&emsp;保持 `venv` 激活,三平台通用:


```bash
    pip install jieba
```


&emsp;&emsp;装完我们就可以在 Python 里 `import jieba` 直接用它来切词了。


### 5.3 用 Pydantic 定义请求/响应模型

&emsp;&emsp;<font color=red>第二步,</font>回到 `analyzer.py`,在最上面追加数据模型的定义。


In [ ]:
from pydantic import BaseModel, Field


class AnalyzeRequest(BaseModel):
    text: str = Field(min_length=1, max_length=10000, description="待分析的中文文本")


class AnalyzeResponse(BaseModel):
    word_count: int          # 分词后的总词数
    char_count: int          # 字符数（含标点）
    top_words: list[tuple[str, int]]   # 词频 Top 5：[(词, 次数), ...]


&emsp;&emsp;这是 `Pydantic` 在本课的**第二次出场**——这次是业务化使用。`AnalyzeRequest` 描述客户端要传给服务端的数据结构,`AnalyzeResponse` 描述服务端要返回给客户端的结构。`Field(min_length=1, max_length=10000)` 自动校验长度,空字符串或超长直接被框架拦截,我们的业务函数根本不会被调用。

&emsp;&emsp;这里有一个很值得注意的细节:`AnalyzeRequest` 里的 `description` 参数,会**自动**出现在 `Swagger UI` 的字段说明里——文档跟代码同步更新,不会出现"代码改了文档没改"的漂移问题。


### 5.4 写三个路由

&emsp;&emsp;<font color=red>第三步,</font>把三个业务路由一次写完。先追加必要的导入和用来模拟数据库的全局存储:


In [ ]:
import jieba
from collections import Counter
from fastapi import HTTPException

HISTORY: dict[int, dict] = {}
NEXT_ID = [1]


&emsp;&emsp;这里每一行都有用意:`jieba` 做中文分词、`Counter` 做词频统计,都是业务逻辑要用的;`HTTPException` 是 `FastAPI` 提供的标准错误抛出方式,比自己拼 `{"error": "..."}` 加状态码省事,且能保证响应格式一致;`HISTORY` 是进程内的"假数据库",用一个 `dict` 存所有分析过的历史记录;`NEXT_ID = [1]` 用列表包裹整数,是为了在函数里递增 `NEXT_ID[0]` 时避免使用 `global` 重新绑定整数。

&emsp;&emsp;> &emsp;<font color=red>注意:</font>`HISTORY = {}` 是进程内全局变量,服务**重启即丢失**——真实项目会用 `SQLAlchemy` / `SQLModel` 接数据库持久化。这里为了聚焦 `FastAPI` 的接口形态,把存储简化掉了。

&emsp;&emsp;**(1)路由 1:`POST /analyze` — 业务核心**


In [ ]:
@app.post("/analyze", response_model=AnalyzeResponse)
def analyze(req: AnalyzeRequest):
    text = req.text
    # jieba 分词，过滤掉单字（"的""了""我"这类），只保留长度 ≥ 2 的有意义词
    words = [w for w in jieba.cut(text) if len(w) >= 2]
    top_words = Counter(words).most_common(5)
    result = {
        "word_count": len(words),
        "char_count": len(text),
        "top_words": top_words,
    }
    HISTORY[NEXT_ID[0]] = {"text": text, "result": result}
    NEXT_ID[0] += 1
    return result


&emsp;&emsp;这段代码的关键点我们逐条看:`req: AnalyzeRequest` 这一行,让 `FastAPI` 把请求体里的 `JSON` 自动解析为 `Pydantic` 对象——已经过校验,进函数体时数据一定合法;`response_model=AnalyzeResponse` 让 `FastAPI` 把返回的 `dict` 按 `AnalyzeResponse` schema 校验并裁剪,多余字段自动剔除,<font color=red>即使函数返回了不该外露的字段也不会泄露</font>;业务逻辑部分是 `jieba` 切词 → 过滤单字 → `Counter.most_common(5)` 取频次前 5,记录写进 `HISTORY`,返回结果。

&emsp;&emsp;**(2)路由 2:`GET /history/{id}` — 按 ID 查历史**


In [ ]:
@app.get("/history/{id}")
def get_history(id: int):
    if id not in HISTORY:
        raise HTTPException(status_code=404, detail="History not found")
    return HISTORY[id]


&emsp;&emsp;`{id}` 写在 `URL` 路径里、函数签名是 `id: int`——`FastAPI` 会自动从 URL 解析成整数(如果传的是 `/history/abc`,会直接返回 422,根本进不到函数体)。查不到时 `raise HTTPException(status_code=404, detail=...)` 抛出标准的 404 响应,而不是自己拼一个 200 + 错误体——这是个关键细节,<font color=red>不能让"没找到"用 200 返回,否则前端没办法走错误分支</font>。

&emsp;&emsp;**(3)路由 3:`GET /search` — 按关键词搜索**


In [ ]:
@app.get("/search")
def search(keyword: str):
    matches = [
        {"id": k, "text": v["text"]}
        for k, v in HISTORY.items()
        if keyword in v["text"]
    ]
    return {"keyword": keyword, "matches": matches}


&emsp;&emsp;`keyword: str` 既没写在路径里、也不是 `Pydantic` 模型——`FastAPI` 会自动认成 `query` 参数(`URL` 里写成 `?keyword=xxx`)。函数体在 `HISTORY` 里做一次 `in` 匹配,返回命中列表。

&emsp;&emsp;这三个路由合起来,覆盖了 `FastAPI` 的三种参数入口:**`Body`**(请求体)——`AnalyzeRequest` 自动从 `JSON body` 解析;**`Path`**(路径参数)——`{id}` 自动从 `URL` 路径解析为 `int`;**`Query`**(查询参数)——`?keyword=xxx` 自动从 `URL query string` 解析。`FastAPI` 的参数推断规则是这样:<font color=red>`Pydantic` 模型 → `Body`;路径里 `{xxx}` → `Path`;其他基本类型 → `Query`</font>。不需要做任何额外配置,写函数签名就够了。


### 5.5 重启 uvicorn 并验证

&emsp;&emsp;<font color=red>第四步,</font>`uvicorn` 检测到代码变化会自动 `reload`。在浏览器地址栏访问 `http://127.0.0.1:8000/docs`,可以看到 4 个路由全部出现(包括最早的 `GET /`),并且每个路由都有完整的请求体 `schema` 和响应体 `schema`。

&emsp;&emsp;**规范自动生成发生了第二次**。回顾贯穿线:

> &emsp;协作需要规范 → `OpenAPI` 是规范标准 → `FastAPI` 把规范自动化 → 浏览器里看到自动规范

&emsp;&emsp;前面讲的所有理论,在这一刻被实证。我们在 `Swagger UI` 里展开 `POST /analyze`,**连续提交三次**。注意下面三条不要一次性粘贴,而是每次填一条、点一次 `Execute`。这样 `HISTORY` 里会有 3 条记录,后面就能按 `id=1/2/3` 取回,也能按关键词搜索:


```text
    第 1 次: {"text": "FastAPI 可以把 Python 函数发布成 API 服务。"}
    第 2 次: {"text": "FastAPI 会根据类型提示自动生成接口文档。"}
    第 3 次: {"text": "Pydantic 会校验请求体,让错误请求自动返回 422。"}
```


&emsp;&emsp;每次响应都会是 `200 OK` + 分析结果。如果服务刚重启,这三次会依次写入 `HISTORY` 的 `id=1/2/3`,三次响应摘要如下:


<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>三次 POST /analyze 的响应摘要</font></p>
<div class="center">

| 提交顺序 | 写入历史 ID | `word_count` | `char_count` | `top_words` 前 5 |
|---|---:|---:|---:|---|
| 第 1 次 | 1 | 7 | 32 | `FastAPI`, `可以`, `Python`, `函数`, `发布` |
| 第 2 次 | 2 | 8 | 24 | `FastAPI`, `根据`, `类型`, `提示`, `自动` |
| 第 3 次 | 3 | 8 | 30 | `请求`×2, `Pydantic`, `校验`, `错误`, `自动` |

</div>


&emsp;&emsp;**再发一个空字符串** `{"text": ""}`,得到 **`422 Unprocessable Entity`** + 整齐的错误 `JSON`:


```json
    {
      "detail": [{
        "type": "string_too_short",
        "loc": ["body", "text"],
        "msg": "String should have at least 1 character"
      }]
    }
```


&emsp;&emsp;`loc` 字段告诉前端**哪个字段错了**——`["body", "text"]` 表示 `body` 里的 `text` 字段。我们<font color=red>没写一行 `if/else`,校验已经发生</font>。422 是后端开发最常见的状态码之一,简单对比一下和它邻近的几个码:

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>常见 HTTP 状态码速查</font></p>
<div class="center">

| 状态码 | 含义 | 典型触发 |
|---|---|---|
| 400 | Bad Request | 通用客户端错误(通常由业务代码主动抛出或自定义处理) |
| 401 | Unauthorized | 没登录 |
| 403 | Forbidden | 登录了但没权限 |
| 404 | Not Found | 路径不存在或资源不存在 |
| 422 | Unprocessable Entity | 请求已进入校验流程,但字段类型或约束没通过(`Pydantic` 校验失败) |
| 500 | Internal Server Error | 服务端崩了 |

</div>


### 5.6 串起三个路由的完整演示

&emsp;&emsp;刚才连续三次 `POST /analyze`,已经在 `HISTORY` 里写下了 `id = 1/2/3` 三条记录。我们用 `Swagger UI` **依次调另外两个路由**,把三种参数入口的行为在一次连续的操作里全部走一遍。

&emsp;&emsp;**(1)调 `GET /history/{id}` 按 ID 取回历史**。展开 `GET /history/{id}`,依次把 `id` 填成 `1`、`2`、`3`,每次点 `Execute`。返回的 `text` 会分别对应刚才三次提交:


<div class="center">

| 填的 `id` | 取回的 `text` |
|---:|---|
| 1 | `FastAPI 可以把 Python 函数发布成 API 服务。` |
| 2 | `FastAPI 会根据类型提示自动生成接口文档。` |
| 3 | `Pydantic 会校验请求体,让错误请求自动返回 422。` |

</div>

&emsp;&emsp;实际响应里还会带上对应的 `result` 分析结果。这里压缩展示 `text`,方便看清楚 `id` 和历史记录的对应关系。


&emsp;&emsp;这是 **`Path` 参数**(`{id}` 自动转成 `int`)的实战。

&emsp;&emsp;**(2)调 `GET /search` 搜出包含 `FastAPI` 的记录**。展开 `GET /search`,`keyword` 填 `FastAPI`,点 `Execute`:


```json
    {
      "keyword": "FastAPI",
      "matches": [
        {"id": 1, "text": "FastAPI 可以把 Python 函数发布成 API 服务。"},
        {"id": 2, "text": "FastAPI 会根据类型提示自动生成接口文档。"}
      ]
    }
```


&emsp;&emsp;`URL` 自动拼成 `http://127.0.0.1:8000/search?keyword=FastAPI`——**`Query` 参数**(`?keyword=xxx`)就是这样自动从 `URL` 解析进函数的 `keyword: str` 参数。

&emsp;&emsp;**(3)再搜另一个关键词 `Pydantic`**。还是展开 `GET /search`,`keyword` 改成 `Pydantic`,点 `Execute`:

```json
    {
      "keyword": "Pydantic",
      "matches": [
        {"id": 3, "text": "Pydantic 会校验请求体,让错误请求自动返回 422。"}
      ]
    }
```

&emsp;&emsp;**(4)故意触发 404**。展开 `GET /history/{id}`,`id` 填 `999`,点 `Execute`:


```json
    {"detail": "History not found"}
```


&emsp;&emsp;状态码是 `404 Not Found`(不是 200),`response body` 里是 `detail` 字段——**这就是 `raise HTTPException(status_code=404, detail=...)` 的标准产物**。前端拿到 404 一眼就能走错误分支,不会和成功响应混淆。

&emsp;&emsp;到这里,三种参数入口(`Body` / `Path` / `Query`)+ 自动校验(422)+ 标准错误(404)全部走过一遍。一个真实的小服务雏形已经成型。


### 5.7 写一个前端页面与跨域问题

&emsp;&emsp;<font color=red>第五步,</font>在 `~/text-analyzer-api/` 下创建 `demo.html`,给我们的后端接口配一个最简单的前端调用页面。


```html
<!DOCTYPE html>
<html>
<head>
  <meta charset="utf-8">
  <title>文本分析 demo</title>
</head>
<body>
  <div style="font-size: 28px; font-weight: 700; margin: 16px 0;">文本分析 demo</div>
  <textarea id="input" rows="6" cols="60"></textarea><br>
  <button onclick="analyze()">分析</button>
  <pre id="output"></pre>
  <script>
    async function analyze() {
      const text = document.getElementById('input').value;
      const resp = await fetch('http://localhost:8000/analyze', {
        method: 'POST',
        headers: { 'Content-Type': 'application/json' },
        body: JSON.stringify({ text })
      });
      const data = await resp.json();
      document.getElementById('output').textContent = JSON.stringify(data, null, 2);
    }
  </script>
</body>
</html>
```


&emsp;&emsp;这段 `HTML` 做的事情非常朴素:一个 `textarea` 输入框、一个按钮,点击后用 `fetch` 调用后端 `/analyze`,把结果渲染到页面下方。我们用 Python 内置的简单服务器跑这个 `HTML`。**新开一个终端窗口**(不要关掉跑 `uvicorn` 的那个窗口),执行:

&emsp;&emsp;`macOS / Linux`(终端):


```bash
    cd ~/text-analyzer-api
    python -m http.server 8001
```


&emsp;&emsp;`Windows`(`PowerShell`):


```powershell
    cd $HOME\text-analyzer-api
    python -m http.server 8001
```


&emsp;&emsp;在浏览器地址栏访问 `http://localhost:8001/demo.html`。输入一段文本"FastAPI 可以把 Python 函数发布成 API 服务。",点击"分析"——**页面没有显示结果**。按 `F12` 打开开发者工具看 `Console` 面板,会看到一行红色报错:


```
    Access to fetch at 'http://localhost:8000/analyze' from origin
    'http://localhost:8001' has been blocked by CORS policy: Response to
    preflight request doesn't pass access control check: No
    'Access-Control-Allow-Origin' header is present on the requested resource.
```


&emsp;&emsp;这是 **`CORS`(跨域资源共享)** 拦截。浏览器有一条强制规则:**如果前端页面的源(`origin`)和它要请求的后端不一致,就要先让后端"明确允许"**。这里前端在 `:8001`,后端在 `:8000`,不同端口在浏览器眼里就算"不同源"。

&emsp;&emsp;有一个容易被忽略的重要事实——**任意非浏览器客户端**(`curl` / `Postman` / `Python requests` / 任何服务端脚本)**都不受这条规则限制**。这是浏览器内部的安全机制,不是后端少返回了什么字段导致的"失败"。理解了这一点,我们就知道"修 CORS"的本质是**让后端明确给浏览器发一个"我允许这个源"的响应头**,而不是改业务代码。


### 5.8 一行 CORSMiddleware 解决跨域

&emsp;&emsp;<font color=red>第六步,</font>回到 `analyzer.py`,在 `app = FastAPI()` 之后加几行中间件配置。


In [ ]:
app.add_middleware(
    CORSMiddleware,
    allow_origins=["http://localhost:8001"],  # 开发时前端的源
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
) 

&emsp;&emsp;`uvicorn` 自动 `reload`。回到浏览器刷新 `demo.html`,按钮可以点了。`Network` 面板会显示:先一个 `OPTIONS preflight` 请求 `200 OK`,然后 `POST /analyze` `200 OK`,返回 `JSON` 渲染在页面上。整条链路跑通。

&emsp;&emsp;<font color=red>生产环境注意:</font>`allow_origins` 必须指定真实前端域名(如 `https://app.example.com`),不能用 `["*"]` 放任。如果同时用 `"*"` 和 `allow_credentials=True`,浏览器会直接拒绝——这是 `CORS` 规范本身的要求。


### 5.9 一个完整应用的形态

&emsp;&emsp;到这里我们已经有了一个完整应用,它包含以下这些部件:**后端 `FastAPI` 服务**(`POST` / `GET` 三个路由 + `Pydantic` 校验)、**前端 `HTML` 页面**(`fetch` 调用接口)、**跨域处理**(`CORSMiddleware`)、**自动文档**(`/docs`)、**完整调用链**(前端 → `CORS` → 路由 → 校验 → 业务逻辑 → 返回 `JSON` → 前端渲染)。

&emsp;&emsp;把 `analyze` 函数里的实现换成任意 Python 能力(`OCR` / 翻译 / 数据查询 / 任何函数),骨架不变——<font color=red>这就是一个可立即改造的通用 API 模板</font>。

&emsp;&emsp;上面这个模板能应付大多数普通后端场景。但碰到 `LLM` 接口——一次调用动辄几秒——必须换一种响应姿势:**流式输出**。下一章我们实现一个真能调通大模型的流式接口,顺便把"`async` 到底解决了什么问题"用一次实验彻底讲清楚。


## 6. 案例二:流式 LLM 接口

&emsp;&emsp;上一章讨论的是普通接口。本章换成 `LLM` 场景。和案例一相比,这里多两件事:一是结果不必等全部生成完再返回,二是请求等待时间更长,需要单独看并发处理。

<div align=center><font size=2 color=#999999>案例二应用全景:前端页面 / 压测页面 → FastAPI 接口 → OpenRouter → 模型服务</font></div>
<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/fastapi-foundations/2026-05-15/img-1db43daa.png" width=80%></div>

<br>


### 6.1 为什么 LLM 接口要走流式

&emsp;&emsp;普通接口常见在毫秒级返回;`LLM` 接口通常是秒级。下面先看一个量级对照表。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>主流 LLM 的响应时长量级</font></p>
<div class="center">

| 模型类别 | TTFT(首 token 时间) | 完整响应(200 token) |
|---|---|---|
| `Claude Haiku` 系列 | 0.5-1s | 2-4s |
| `GPT-4o-mini` | 0.5-1s | 2-4s |
| `Claude Sonnet` 系列 | 1-2s | 5-10s |
| `DeepSeek-V3` 系列 | 1-3s | 5-15s |
| 推理模型(`o3` / `DeepSeek-R1`) | 10-30s | 30-120s |

</div>

&emsp;&emsp;当响应持续数秒时,如果后端一直等到完整结果再返回,前端只能持续等待,交互反馈会比较弱。对于 `LLM` 接口,更常见的处理方式是边生成边返回。

&emsp;&emsp;也就是**流式响应(`Streaming`)**:服务端边生成边发送,客户端边接收边显示。技术上可以直接基于 `HTTP` 响应流完成,不一定需要额外协议。下面把这件事落实到代码。


### 6.2 建一个新项目

&emsp;&emsp;<font color=red>第一步,</font>案例二单独使用一个目录和虚拟环境,和案例一的 `text-analyzer-api` 隔离开。

&emsp;&emsp;案例二单独放在一个目录里,依赖和环境变量也单独管理。这样后面运行和排错会更方便。

&emsp;&emsp;新开一个终端窗口(不要在 `text-analyzer-api/` 里),`macOS / Linux`:


```bash
    cd ~
    mkdir llm-stream-api
    cd llm-stream-api
    python3 -m venv venv
    source venv/bin/activate
    pip install "fastapi[standard]" openai python-dotenv
```


&emsp;&emsp;`Windows`(`PowerShell`):


```powershell
    cd $HOME
    mkdir llm-stream-api
    cd llm-stream-api
    py -m venv venv
    .\venv\Scripts\Activate.ps1
    pip install "fastapi[standard]" openai python-dotenv
```


&emsp;&emsp;这一步完成四件事:建目录、进入目录、创建新的 `venv`、安装依赖。其中 `openai` 是官方 `SDK`,`python-dotenv` 用来加载 `.env` 里的环境变量。


### 6.3 放 .env

&emsp;&emsp;<font color=red>第二步,</font>配置 `API Key`。这里使用 `OpenRouter`,因为它提供 `OpenAI` 兼容接口,后面切换模型时只需要改 `model=`。

&emsp;&emsp;去 [openrouter.ai/keys](https://openrouter.ai/keys) 申请一个 `API Key`,然后在 `~/llm-stream-api/` 下创建 `.env`:


```
    OPENROUTER_API_KEY=sk-or-v1-你的密钥
```


&emsp;&emsp;<font color=red>`.env` 用来放敏感配置,不要把 key 直接写进 `stream.py`</font>。另外,`.env` 本身要加进 `.gitignore`。


### 6.4 写 stream.py

&emsp;&emsp;<font color=red>第三步,</font>在 `~/llm-stream-api/` 下创建 `stream.py`——这是我们这个案例的全部服务端代码。


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()  # 启动时自动从 .env 加载环境变量

from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import StreamingResponse
from pydantic import BaseModel, Field
from openai import AsyncOpenAI

app = FastAPI()
app.add_middleware(
    CORSMiddleware,
    allow_origins=["http://127.0.0.1:8001", "http://localhost:8001"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# 异步版 OpenAI 客户端，base_url 指向 OpenRouter
llm = AsyncOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)


class ChatRequest(BaseModel):
    message: str = Field(min_length=1, description="用户输入")


@app.post("/chat/stream")
async def chat_stream(req: ChatRequest):
    async def generate():
        stream = await llm.chat.completions.create(
            model="qwen/qwen-turbo",
            messages=[{"role": "user", "content": req.message}],
            stream=True,    # 关键：开启 OpenAI SDK 的流式模式
        )
        async for chunk in stream:
            if not chunk.choices:    # 末尾的 usage-only chunk，跳过
                continue
            content = chunk.choices[0].delta.content or ""
            if content:
                yield content
    return StreamingResponse(generate(), media_type="text/plain")


&emsp;&emsp;这段代码先看几个位置。导入部分里,`FastAPI` 用来定义路由,`CORSMiddleware` 处理跨域,`StreamingResponse` 返回流式响应,`ChatRequest` 做请求体校验,`AsyncOpenAI` 调上游模型。

&emsp;&emsp;入口是 `@app.post("/chat/stream")`。`req: ChatRequest` 表示请求体按这个模型解析,函数里直接用 `req.message`。这里加 `CORSMiddleware`,是因为后面的页面 `llm_stream.html` 跑在 `8001`,接口跑在 `8000`,浏览器会把它视为跨域请求。

&emsp;&emsp;流式输出从两处开始。第一处是 `stream=True`,它告诉上游不要等完整答案,而是分段返回。第二处是 `async for chunk in stream` 配合 `yield content`:每读到一段新文本,就立即向下游发送一段。

&emsp;&emsp;最后 `return StreamingResponse(generate(), media_type="text/plain")` 会把生成器里的内容持续写回浏览器。前端再用 `response.body.getReader()` 按段读取,就能看到文本逐步追加。这里先用 `qwen/qwen-turbo`;如果后面要换模型,改 `model=` 即可。


### 6.5 启动服务并用前端页面看流式效果

&emsp;&emsp;这一节直接用浏览器页面 `llm_stream.html` 来看返回过程。这样 `Windows / macOS` 的操作步骤一致,也更容易看出内容是不是分段到达。

&emsp;&emsp;<font color=red>第四步,</font>启动服务。保持在 `llm-stream-api/` 目录、`venv` 激活,三平台通用:


```bash
    uvicorn stream:app --reload
```


&emsp;&emsp;<font color=red>第五步,</font>在 `~/llm-stream-api/` 下放一个 `llm_stream.html`。这个页面只做一件事:把输入框内容发给 `/chat/stream`,然后用 `response.body.getReader()` 一段段读取流式响应。


&emsp;&emsp;完整页面文件不在正文里展开——直接使用同目录的 `llm_stream.html` 即可。正文只保留最关键的流式读取片段,其他样式、状态条、快捷提示词和停止按钮都已经封装在页面文件里。

```html
<script>
async function startStream() {
  const response = await fetch("http://127.0.0.1:8000/chat/stream", {
    method: "POST",
    headers: { "Content-Type": "application/json" },
    body: JSON.stringify({ message: messageEl.value.trim() }),
    signal: controller.signal,
  });

  const reader = response.body.getReader();
  const decoder = new TextDecoder("utf-8");

  while (true) {
    const { done, value } = await reader.read();
    if (done) break;
    outputEl.textContent += decoder.decode(value, { stream: true });
  }
}
</script>
```

&emsp;&emsp;这个逻辑的关键只有三步:第一,`fetch` 发出 `POST /chat/stream`;第二,`response.body.getReader()` 拿到流式读取器;第三,`TextDecoder` 把每一段字节解码成文本并追加到页面。浏览器里看到的"字一段一段出现",本质上就是这个循环在不断消费服务端的流。


&emsp;&emsp;<font color=red>第六步,</font>新开一个终端窗口,保持在 `llm-stream-api/` 目录,启动一个最简单的静态文件服务器。这样浏览器能用 `http://127.0.0.1:8001/llm_stream.html` 正常访问页面。`macOS / Linux / Windows` 在 `venv` 激活后都能直接跑:


```bash
    python -m http.server 8001

    # 如果 Windows 里 python 命令不可用，就用：
    # py -m http.server 8001
```


&emsp;&emsp;浏览器打开 `http://127.0.0.1:8001/llm_stream.html`。在输入框里写"用三句话介绍 FastAPI",点击"开始生成"后,右侧输出区会逐步增长。如果页面能持续追加文本,说明 `StreamingResponse` 已经工作起来。因为页面跑在 `8001`、接口跑在 `8000`,这里也会用到前面加的 `CORSMiddleware`。

&emsp;&emsp;页面里还放了三个辅助元素:提示词按钮、状态条和停止按钮。它们分别对应快速填充示例输入、显示当前状态以及中断尚未结束的请求。


### 6.6 async vs sync 的并发表现

&emsp;&emsp;到这里 `/chat/stream` 已经可以工作。单个请求时,`async` 和 `sync` 往往都能返回结果;差别主要出现在并发时。等待型任务用 `async` 可以在同一事件循环里交替推进,同步等待则会占住线程,并发高了以后会排队。

&emsp;&emsp;如果直接拿真实 `LLM` 做对比,结果会同时受到上游波动、限速和费用影响,不容易说明问题。

&emsp;&emsp;所以这里把上游调用换成固定的等待逻辑:`asyncio.sleep(5)` 和 `time.sleep(5)`。两边总等待时间一致,只比较等待方式不同带来的并发表现。

&emsp;&emsp;**(1)把并发对比接口补到 `stream.py`**

&emsp;&emsp;这里不额外新建 `py` 文件,直接在 `~/llm-stream-api/stream.py` 里补一段 `/chat/benchmark` 代码即可。


In [ ]:
import asyncio
import time
from typing import Literal

import anyio
from pydantic import BaseModel, Field


class BenchmarkRequest(BaseModel):
    mode: Literal["async", "sync"]
    concurrency: int = Field(default=100, ge=1, le=300)


async def _simulate_async_job():
    for _ in range(10):
        await asyncio.sleep(0.5)   # ← 关键：异步 sleep，让出事件循环


def _simulate_sync_job():
    for _ in range(10):
        time.sleep(0.5)            # ← 关键：同步 sleep，阻塞线程


@app.post("/chat/benchmark")
async def chat_benchmark(req: BenchmarkRequest):
    started_at = time.perf_counter()

    if req.mode == "async":
        await asyncio.gather(*(_simulate_async_job() for _ in range(req.concurrency)))
    else:
        await asyncio.gather(
            *(anyio.to_thread.run_sync(_simulate_sync_job) for _ in range(req.concurrency))
        )

    return {
        "mode": req.mode,
        "concurrency": req.concurrency,
        "completed": req.concurrency,
        "elapsed_seconds": round(time.perf_counter() - started_at, 2),
    }


&emsp;&emsp;这个 `benchmark` 接口比较的是两种等待方式:

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>async 与 sync 模式的变量隔离</font></p>
<div class="center">

| 对比点 | async 模式 | sync 模式 |
|---|---|---|
| 对外入口 | `POST /chat/benchmark` + `mode="async"` | `POST /chat/benchmark` + `mode="sync"` |
| 内部等待 | `await asyncio.sleep(0.5)` —— **让出事件循环** | `time.sleep(0.5)` —— **阻塞当前线程** |

</div>

&emsp;&emsp;对外只保留 `/chat/benchmark` 一个入口,`async` 和 `sync` 通过参数切换。把代码保存到 `stream.py` 后,如果第 4 步的 `uvicorn stream:app --reload` 还在运行,它会自动热重载;如果已经停掉,就重新启动一次:


```bash
    uvicorn stream:app --reload
```


&emsp;&emsp;现在 `stream.py` 对外保留两个接口:`/chat/stream` 和 `/chat/benchmark`。下一步直接在页面里切换 `async` / `sync` 模式即可。

&emsp;&emsp;**(2)跑 100 并发对比**

&emsp;&emsp;<font color=red>为什么选 100?</font>`Starlette` 底层会通过 `AnyIO` 线程池处理同步等待逻辑,默认容量限制是 **40 个 tokens**。把并发数设为 100,是为了明显超过这个默认容量,从而观察排队现象;如果并发数只有 50,超出部分较少,差异通常没有这么明显。

&emsp;&emsp;这里同样使用页面 `async_sync.html`。页面本身只发一次 `/chat/benchmark` 请求,100 个任务由服务端内部创建,这样结果不会被浏览器同源连接上限影响。


```html
<script>
const selectedMode = "async";
const response = await fetch(`${API_BASE}/chat/benchmark`, {
    method: "POST",
    headers: { "Content-Type": "application/json" },
    body: JSON.stringify({ mode: selectedMode, concurrency }),
});

const data = await response.json();
const elapsed = data.elapsed_seconds;
</script>
```


&emsp;&emsp;完整页面文件同样不在正文里展开——直接使用同目录的 `async_sync.html` 即可。这里保留的关键点只有一个:浏览器只负责触发一次 `/chat/benchmark`,并发任务在服务端内部完成。


```bash
    # 如果前一节的静态文件服务器还在跑,这一节不用再启动一次
    python -m http.server 8001
```


&emsp;&emsp;浏览器打开 `http://127.0.0.1:8001/async_sync.html`。并发数保持 `100`,先点一次"测试 async",再点一次"测试 sync"。页面会把每次的总耗时记到右侧表格里。


```
    async | 100 并发 | 5.01 秒
```


&emsp;&emsp;这里测得 `5.01` 秒,基本接近单次任务的 5 秒。原因是 100 个任务都在同一事件循环里等待,等待阶段可以重叠。

&emsp;&emsp;再点一次"测试 sync"。这里不需要重启服务,只是把 `/chat/benchmark` 的 `mode` 改成 `sync`。


```
    sync | 100 并发 | 15.12 秒
```


&emsp;&emsp;页面实测示例(不同机器会有少量浮动):


```
    sync | 100 并发 | 15.12 秒
```


&emsp;&emsp;这里测得 `15.12` 秒,明显高于 `async`。切到 `sync` 模式后,任务会占用 `AnyIO` 线程池;当前面的线程用满时,后面的任务只能继续排队。

&emsp;&emsp;这里也不是 `100 × 5 = 500` 秒。`FastAPI / Starlette` 处理同步等待时,会借助线程池并行推进一部分任务,只是容量有限,超过之后才开始排队。

&emsp;&emsp;**(3)结果对比**

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>100 并发下 async vs sync 实测</font></p>
<div class="center">

| 模式 | 内部等待方式 | 100 并发实测耗时 | 相对速度 |
|---|---|---|---|
| `async` | `asyncio.sleep` **让出事件循环** | **5.01 秒** | 基线(1×) |
| `sync` | `time.sleep` **阻塞线程** | **15.12 秒** | **3.02×** 慢 |

</div>

&emsp;&emsp;在这个实验设置下,`async` 的耗时接近单次任务时长;`sync` 在超过线程池容量后会继续变慢。两者的主要差别就在这里。

&emsp;&emsp;从这个实验可以得到一个简单结论:如果任务主要是在等待 `I/O`,并且并发开始明显高于默认线程池容量,`async` 模式通常更适合。真实系统的容量仍然需要结合压测、worker 配置和上游限流一起判断。


### 6.7 回看自动文档与校验

&emsp;&emsp;实验跑完,直接打开 `http://127.0.0.1:8000/docs`,会看到 `/chat/stream` 和 `/chat/benchmark` 两个接口,请求体 `schema` 也会自动生成出来(`ChatRequest`、`BenchmarkRequest`)。

&emsp;&emsp;到这里,这一章的主线可以再回看一次:

> &emsp;协作需要规范 → `OpenAPI` 是规范标准 → 类型提示和 `Pydantic` 描述请求结构 → `FastAPI` 自动生成文档、规范和校验

&emsp;&emsp;这里的"自动"主要是三件事:第一,`/docs` 里自动出现可交互接口文档;第二,`/openapi.json` 里自动生成机器可读的接口规范;第三,请求体不符合 `ChatRequest` 这类 `Pydantic` 模型时,自动返回 `422`。无论是文本分析还是 `LLM` 流式返回,入口处的请求结构、接口文档和参数校验都由同一套类型提示 + `Pydantic` 机制完成。下一章我们把本课的收获与速查一并整理清楚。


## 7. 本课小结与速查表

&emsp;&emsp;到这里,两个完整案例都跑通了。本章把本课的收获按"成品代码、常见报错、速查表"三条线沉淀下来,方便之后回查。


### 7.1 本课收获

&emsp;&emsp;走完整个过程,本地现在有两组可运行代码。`~/text-analyzer-api/analyzer.py` 和 `~/text-analyzer-api/demo.html` 对应普通接口与前端调用;`~/llm-stream-api/stream.py`、`~/llm-stream-api/llm_stream.html`、`~/llm-stream-api/async_sync.html` 对应流式接口、并发对比接口和两个页面。后续把 `analyze` 或 `chat_stream` 里的业务逻辑换成自己的实现,就可以继续扩展。

&emsp;&emsp;同时,**几种常见报错的排查思路**也已经打通,以后再遇到这些场景,脑子里会有清晰的因果链。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>常见报错场景与排查思路</font></p>
<div class="center">

| 看到... | 就知道... | 怎么修 |
|---|---|---|
| `422 Unprocessable Entity` | `Pydantic` 校验失败 | 看响应里 `loc` 字段定位哪个字段哪种错 |
| `404 Not Found` | 资源不存在 | 用 `raise HTTPException(status_code=404)` 标准抛错 |
| 浏览器 `CORS` 红屏 | 跨域安全规则没通过 | 后端加 `CORSMiddleware` 明确允许前端的源 |
| `LLM` 接口卡很久不出东西 | 调用本身要几秒 | 用 `StreamingResponse` + `stream=True` 做流式输出;AI 接口默认写 `async def + AsyncOpenAI` |

</div>

&emsp;&emsp;接下来的几小节是速查表附录,覆盖 `Pydantic v2 Field` 约束、`HTTP` 状态码、路由参数三种入口、`CORSMiddleware` 配置、`uvicorn` 启动命令、`async` 使用决策——用的时候翻回来即可。


### 7.2 Pydantic v2 Field 约束速查

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>Pydantic v2 常用 Field 约束</font></p>
<div class="center">

| 约束 | 作用 | 示例 |
|---|---|---|
| `min_length` / `max_length` | 字符串/列表长度 | `Field(min_length=1, max_length=100)` |
| `pattern` | 正则匹配 | `Field(pattern=r"^\w+@\w+$")` |
| `gt` / `ge` / `lt` / `le` | 数值大小 | `Field(gt=0, le=100)` |
| `default` | 默认值 | `Field(default="anonymous")` |
| `default_factory` | 工厂默认值 | `Field(default_factory=list)` |
| `description` | 字段说明(出现在 `/docs`) | `Field(description="用户名")` |
| 可选字段 | `str \| None = None` | (Python 3.10+ 联合类型语法) |

</div>


### 7.3 HTTP 状态码速查

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>常用 HTTP 状态码</font></p>
<div class="center">

| 状态码 | 含义 | 何时用 |
|---|---|---|
| 200 | OK | 标准成功 |
| 201 | Created | `POST` 创建成功 |
| 204 | No Content | `DELETE` 成功无返回体 |
| 400 | Bad Request | 通用客户端请求错误;需要业务主动返回时用 |
| 401 | Unauthorized | 没登录 |
| 403 | Forbidden | 登录了但没权限 |
| 404 | Not Found | 资源不存在,`raise HTTPException(status_code=404)` |
| 422 | Unprocessable Entity | 请求体/参数被 `FastAPI + Pydantic` 解析或校验失败 |
| 500 | Internal Server Error | 后端崩了 |
| 503 | Service Unavailable | 服务暂时不可用 |

</div>


### 7.4 路由参数三种入口

&emsp;&emsp;一个路由函数可以同时从三种位置拿参数:路径、`query string`、请求体。`FastAPI` 会根据参数的类型自动推断来源,写函数签名就够了。

<div align='center'>
<svg width='900' height='330' viewBox='0 0 900 330' xmlns='http://www.w3.org/2000/svg' role='img' aria-label='FastAPI 路由参数三种入口示意图' style='max-width:100%;height:auto;'>
<defs>
  <marker id='route-arrow' markerWidth='10' markerHeight='10' refX='9' refY='5' orient='auto' markerUnits='strokeWidth'>
    <path d='M 0 0 L 10 5 L 0 10 z' fill='#3b82f6'/>
  </marker>
  <style>
    .title{font:700 20px -apple-system,BlinkMacSystemFont,'Segoe UI',Arial,sans-serif;fill:#111827;}
    .label{font:700 15px -apple-system,BlinkMacSystemFont,'Segoe UI',Arial,sans-serif;fill:#1f2937;}
    .txt{font:14px -apple-system,BlinkMacSystemFont,'Segoe UI',Arial,sans-serif;fill:#4b5563;}
    .code{font:13px Menlo,Consolas,'Courier New',monospace;fill:#1f2937;}
    .small{font:12px -apple-system,BlinkMacSystemFont,'Segoe UI',Arial,sans-serif;fill:#6b7280;}
  </style>
</defs>
<rect x='12' y='12' width='876' height='306' rx='8' fill='#f8fbff' stroke='#dbeafe'/>
<text x='450' y='42' text-anchor='middle' class='title'>一次 HTTP 请求里的三种参数入口</text>

<rect x='40' y='82' width='250' height='170' rx='8' fill='#ffffff' stroke='#bfdbfe'/>
<text x='165' y='112' text-anchor='middle' class='label'>客户端请求</text>
<text x='65' y='145' class='code'>POST /items/123?keyword=fastapi</text>
<text x='65' y='178' class='code'>Body: { ... }</text>
<text x='65' y='213' class='small'>路径、查询字符串、请求体都在同一次请求里</text>

<rect x='350' y='70' width='210' height='64' rx='8' fill='#eff6ff' stroke='#93c5fd'/>
<text x='370' y='96' class='label'>Path</text>
<text x='370' y='119' class='code'>/items/{item_id} → item_id</text>

<rect x='350' y='146' width='210' height='64' rx='8' fill='#f0fdf4' stroke='#86efac'/>
<text x='370' y='172' class='label'>Body</text>
<text x='370' y='195' class='code'>Pydantic 模型 → body</text>

<rect x='350' y='222' width='210' height='64' rx='8' fill='#fff7ed' stroke='#fdba74'/>
<text x='370' y='248' class='label'>Query</text>
<text x='370' y='271' class='code'>?keyword=xxx → keyword</text>

<rect x='640' y='78' width='220' height='182' rx='8' fill='#ffffff' stroke='#c7d2fe'/>
<text x='750' y='108' text-anchor='middle' class='label'>路由函数签名</text>
<text x='665' y='140' class='code'>def example(</text>
<text x='685' y='165' class='code'>item_id: int,</text>
<text x='685' y='190' class='code'>body: ItemRequest,</text>
<text x='685' y='215' class='code'>keyword: str = &quot;&quot;,</text>
<text x='665' y='240' class='code'>): ...</text>

<line x1='290' y1='135' x2='350' y2='102' stroke='#3b82f6' stroke-width='2' marker-end='url(#route-arrow)'/>
<line x1='290' y1='168' x2='350' y2='178' stroke='#22c55e' stroke-width='2' marker-end='url(#route-arrow)'/>
<line x1='290' y1='201' x2='350' y2='254' stroke='#f97316' stroke-width='2' marker-end='url(#route-arrow)'/>
<line x1='560' y1='102' x2='640' y2='160' stroke='#3b82f6' stroke-width='2' marker-end='url(#route-arrow)'/>
<line x1='560' y1='178' x2='640' y2='187' stroke='#22c55e' stroke-width='2' marker-end='url(#route-arrow)'/>
<line x1='560' y1='254' x2='640' y2='212' stroke='#f97316' stroke-width='2' marker-end='url(#route-arrow)'/>
</svg>
</div>
<div align='center'><font size='2' color='#999999'>FastAPI 根据函数签名把 Path / Body / Query 自动映射到参数</font></div>


In [ ]:
@app.post("/items/{item_id}")
def example(
    item_id: int,                    # Path：路径中的 {item_id}
    body: ItemRequest,               # Body：Pydantic 模型自动从 JSON body 解析
    keyword: str = "",               # Query：URL 后 ?keyword=xxx
):
    ...


&emsp;&emsp;推断规则是这样的:<font color=red>`Pydantic` 模型 → `Body`;路径里 `{x}` → `Path`;其他基本类型 → `Query`</font>。不需要任何装饰器参数额外指定。


### 7.5 CORSMiddleware 配置

&emsp;&emsp;开发环境和生产环境的 `CORSMiddleware` 配置应当不同——开发环境为了调方便可以放宽,生产环境必须收紧到真实域名。

&emsp;&emsp;**开发环境:**


In [ ]:
app.add_middleware(
    CORSMiddleware,
    allow_origins=["http://localhost:8001"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)


&emsp;&emsp;**生产环境:**


In [ ]:
app.add_middleware(
    CORSMiddleware,
    allow_origins=["https://app.example.com"],   # 必须明确域名
    allow_credentials=True,
    allow_methods=["GET", "POST"],               # 收紧
    allow_headers=["Content-Type", "Authorization"],
)


&emsp;&emsp;<font color=red>禁忌:</font>`allow_origins=["*"]` + `allow_credentials=True` 浏览器会直接拒绝——这是 `CORS` 规范本身的要求,不是某个浏览器实现的特例。


### 7.6 uvicorn 启动命令

&emsp;&emsp;命令里的 `<文件>:app` 格式是 `Python 文件名(不带 .py):FastAPI 实例对象名`——本课案例一里写成 `analyzer:app`,案例二里写成 `stream:app`,下面统一写成 `<文件>:app` 占位。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>常用 uvicorn 启动命令</font></p>
<div class="center">

| 场景 | 命令 |
|---|---|
| 开发(自动 `reload`) | `uvicorn <文件>:app --reload` |
| 开发(指定端口) | `uvicorn <文件>:app --reload --port 8080` |
| 生产(多 `worker`) | `uvicorn <文件>:app --workers 4 --host 0.0.0.0 --port 80` |
| 生产(用 `gunicorn` 管理 `uvicorn worker`) | 先 `pip install gunicorn uvicorn-worker`,再 `gunicorn -k uvicorn_worker.UvicornWorker -w 4 <文件>:app` |

</div>


### 7.7 async 使用决策

&emsp;&emsp;什么时候写 `async def`、什么时候写 `def`?一张决策树帮你快速判断。


```
    路由函数体里的代码主要做什么:
    ├── 调用 LLM / 网络 IO / 数据库 IO 且对应库支持 await(如 httpx, asyncpg, AsyncOpenAI)
    │       → 用 async def + await
    ├── CPU 密集(大段计算 / 图像处理 / 模型推理)
    │       → 短任务可用 def 避免占事件循环;重 CPU 任务应考虑进程池 / 任务队列 / 独立模型服务
    ├── 调用同步 IO 库(如 requests, psycopg2, OpenAI 同步版)
    │       → 用 def 或线程池封装;高并发场景优先换 async 客户端
    └── 简单逻辑(dict 操作、字符串处理)
            → 用 def 即可,没必要 async
```


&emsp;&emsp;<font color=red>经验规则:</font>`async def` 函数体里不要直接调用 `time.sleep`、`requests.get` 这类同步阻塞函数——它们会卡住事件循环。必须用同步库时,放到 `def` 路由、线程池封装,或换成对应的异步客户端。

&emsp;&emsp;**版本说明:**本课示例按 `FastAPI 0.115+`、`Pydantic 2.x`、`Python 3.10+` 编写,并在 2026-04 按当前官方文档口径复核过;若未来版本调整命令或默认值,以官方文档为准。到这里,本课的所有内容全部交付完毕。把 `analyzer.py` 和 `stream.py` 两份代码改成你业务里的实际能力,就是你自己项目的最小可跑后端。
